# Recomendador de películas mediante similitud de documentos

Los sistemas de recomendación son una de las aplicaciones más populares y adoptadas del aprendizaje automático. Normalmente se utilizan para recomendar entidades a los usuarios, y estas entidades pueden ser productos, películas, servicios, entre otros.

Ejemplos populares de recomendaciones incluyen:
	•	Amazon sugiriendo productos en su sitio web
	•	Amazon Prime, Netflix, Hotstar recomendando películas o series
	•	YouTube recomendando videos para ver

Típicamente, los sistemas de recomendación se pueden implementar de tres formas:

1.	Recomendadores basados en reglas simples: Generalmente se basan en métricas y umbrales globales, como la popularidad de una película, calificaciones globales, etc.
2.	Recomendadores basados en contenido: Se basan en proporcionar entidades similares a una entidad específica de interés. Aquí se puede usar metadatos del contenido como descripciones de películas, género, reparto, director, entre otros.
3.	Recomendadores basados en filtrado colaborativo: En este caso no se requieren metadatos, sino que se intenta predecir recomendaciones y calificaciones en función de las calificaciones pasadas de diferentes usuarios y elementos específicos.


![](https://storage.googleapis.com/kaggle-datasets-images/138/287/229bfb5d3dd1a49cc5ac899c45ca2213/dataset-cover.png)

Dado que nuestro enfoque no son propiamente los motores de recomendación, sino el procesamiento de lenguaje natural (NLP), utilizaremos los metadatos basados en texto de cada película para intentar recomendar películas similares basándonos en películas específicas de interés. Esto se clasifica dentro de los recomendadores basados en contenido.

## Dataset IMDB

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('tmdb_5000_movies.csv.gz', compression='gzip')
df.info()

In [ ]:
df.head()

In [ ]:
df = df[['title', 'tagline', 'overview', 'popularity']]
df.tagline.fillna('', inplace=True)
df['description'] = df['tagline'].map(str) + ' ' + df['overview']
df.dropna(inplace=True)
df = df.sort_values(by=['popularity'], ascending=False)
df.info()

In [ ]:
df.head()

## Construcción un Sistema Recomendador de Películas

Para construir un sistema recomendador de películas. Utilizaremos el siguiente flujo de trabajo:

1.	Preprocesamiento de texto
2.	Ingeniería de características
3.	Cálculo de similitud entre documentos
4.	Búsqueda de las películas más similares
5.	Construcción de una función de recomendación de películas

In [ ]:
import nltk
import re
import numpy as np
import contractions

stop_words = nltk.corpus.stopwords.words('english')

def normalize_document(doc):
    # lower case and remove special characters\whitespaces
    doc = re.sub(r'[^a-zA-Z0-9\s]', '', doc, re.I|re.A)
    doc = doc.lower()
    doc = doc.strip()
    doc = contractions.fix(doc)
    # tokenize document
    tokens = nltk.word_tokenize(doc)
    #filter stopwords out of document
    filtered_tokens = [token for token in tokens if token not in stop_words]
    # re-create document from filtered tokens
    ##doc = ' '.join(filtered_tokens)
    return filtered_tokens


In [ ]:
df['normalized'] = df['description'].apply(normalize_document)

In [ ]:
df.head()

## Vectorización one-hot

La vectorización por one-hot puede ser realizada con ayuda de scikit-learn mediante la clase `NultiLabelBinarizer` (No utilizar `OneHotEncoder` que es utilizado para características categóricas).

Mediante el método `.fit()` se aprende el vocabulario, y con el método `transform()` convierte los documentos a vectores. 

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

lb = MultiLabelBinarizer()
onehot = lb.fit_transform(df['normalized'])

In [ ]:
lb.classes_

### Matriz Documento-Término

La matriz de documento-término (Document-Term Matrix) se compone agregando como filas todos los documentos vectorizados. Por lo que, representa a todos los documentos vectorizados.

In [ ]:
pd.DataFrame(onehot, columns=list(lb.classes_))

## Matriz de similitud

Para encontrar la similitud de todos los documentos, se puede obtener a partir de la generalización de la fórmula:

$s_{ij} = d_i \cdot d_j$

Si se desea utilizar la matriz de documento-término para después, se puede escribir el producto escalar como una suma:

$s_{ij} = \sum_k D_{ik} D_{jk} = \sum_k D_{ik} (D^T)_{jk} = (D \cdot D^T)_{ij} $

In [ ]:
doc_sim = np.dot(onehot, np.transpose(onehot))

In [ ]:
doc_sim_df = pd.DataFrame(doc_sim)
doc_sim_df.head()

### Obtener la lista de todas las películas

In [ ]:
movies_list = df['title'].values
movies_list, movies_list.shape

### Encontrar las películas más similares para una película de ejemplo

Tomemos Minions, la película más popular del dataframe anterior, e intentemos encontrar las películas más similares que puedan recomendarse.

In [ ]:
movie_idx = np.where(movies_list == 'Minions')[0][0]
movie_idx

In [ ]:
movie_similarities = doc_sim_df.iloc[movie_idx].values
movie_similarities

In [ ]:
similar_movie_idxs = np.argsort(-movie_similarities)[1:6]
similar_movie_idxs

In [ ]:
similar_movies = movies_list[similar_movie_idxs]
similar_movies

## Construir una función recomendadora de películas que sugiera las 5 más similares para cualquier película

El título de la película, la lista de títulos de películas y el dataframe de la matriz de similitud de documentos se proporcionarán como entradas de la función.

In [ ]:
def movie_recommender(movie_title, movies=movies_list, doc_sims=doc_sim_df):
    pass

In [ ]:
popular_movies = ['Minions', 'Interstellar', 'Deadpool', 'Jurassic World', 'Pirates of the Caribbean: The Curse of the Black Pearl',
              'Dawn of the Planet of the Apes', 'The Hunger Games: Mockingjay - Part 1', 'Terminator Genisys', 
              'Captain America: Civil War', 'The Dark Knight', 'The Martian', 'Batman v Superman: Dawn of Justice', 
              'Pulp Fiction', 'The Godfather', 'The Shawshank Redemption', 'The Lord of the Rings: The Fellowship of the Ring',  
              'Harry Potter and the Chamber of Secrets', 'Star Wars', 'The Hobbit: The Battle of the Five Armies',
              'Iron Man']

In [ ]:
for movie in popular_movies:
    print('Movie:', movie)
    print('Top 5 recommended Movies:', movie_recommender(movie_title=movie, movies=movies_list, doc_sims=doc_sim_df))
    print()